In [1]:
import glob
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import numpy as np
from regions import Regions
from astropy.nddata import Cutout2D
import astropy.units as u
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from astropy.table import Table, vstack
from astropy.coordinates import match_coordinates_sky
from astropy.coordinates import SkyCoord
from astropy.io import fits
import os
import time
import datetime
import warnings
from astropy.table import MaskedColumn
from astropy import table
from astroquery.svo_fps import SvoFps
from tqdm.auto import tqdm

image_filenames ={
    "f140m": "/orange/adamginsburg/jwst/w51/F140M/pipeline/jw06151-o001_t001_nircam_clear-f140m-merged_i2d.fits",
    "f150w": "/orange/adamginsburg/jwst/w51/F150W/pipeline/jw06151-o001_t001_nircam_clear-f150w-merged_i2d.fits",
    "f162m": "/orange/adamginsburg/jwst/w51/F162M/pipeline/jw06151-o001_t001_nircam_clear-f162m-merged_i2d.fits",
    "f182m": "/orange/adamginsburg/jwst/w51/F182M/pipeline/jw06151-o001_t001_nircam_clear-f182m-merged_i2d.fits",
    "f187n": "/orange/adamginsburg/jwst/w51/F187N/pipeline/jw06151-o001_t001_nircam_clear-f187n-merged_i2d.fits",
    "f210m": "/orange/adamginsburg/jwst/w51/F210M/pipeline/jw06151-o001_t001_nircam_clear-f210m-merged_i2d.fits",
    "f335m": "/orange/adamginsburg/jwst/w51/F335M/pipeline/jw06151-o001_t001_nircam_clear-f335m-merged_i2d.fits",
    "f360m": "/orange/adamginsburg/jwst/w51/F360M/pipeline/jw06151-o001_t001_nircam_clear-f360m-merged_i2d.fits",
    "f405n": "/orange/adamginsburg/jwst/w51/F405N/pipeline/jw06151-o001_t001_nircam_clear-f405n-merged_i2d.fits",
    "f410m": "/orange/adamginsburg/jwst/w51/F410M/pipeline/jw06151-o001_t001_nircam_clear-f410m-merged_i2d.fits", # weird, the filename is different from what is downloaded with the STScI pipeline...
    "f480m": "/orange/adamginsburg/jwst/w51/F480M/pipeline/jw06151-o001_t001_nircam_clear-f480m-merged_i2d.fits",
    "f560w": "/orange/adamginsburg/jwst/w51/F560W/pipeline/jw06151-o002_t001_miri_f560w_i2d.fits",
    "f770w": "/orange/adamginsburg/jwst/w51/F770W/pipeline/jw06151-o002_t001_miri_f770w_i2d.fits",
    "f1000w": "/orange/adamginsburg/jwst/w51/F1000W/pipeline/jw06151-o002_t001_miri_f1000w_i2d.fits",
    "f1280w": "/orange/adamginsburg/jwst/w51/F1280W/pipeline/jw06151-o002_t001_miri_f1280w_i2d.fits",
    "f1500w": "/orange/adamginsburg/jwst/w51/F1500W/pipeline/jw06151-o002_t001_miri_f1500w_i2d.fits",
    "f2100w": "/orange/adamginsburg/jwst/w51/F2100W/pipeline/jw06151-o002_t001_miri_f2100w_i2d.fits",
    
}



def merge_nrca_nrcb(filt):
    nrca_catalog = Table.read(f'/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/{filt.lower()}_nrca_indivexp_merged_dao_after_merger_combined_with_satstars_nmatch_cut_grade_a.fits')
    nrcb_catalog = Table.read(f'/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/{filt.lower()}_nrcb_indivexp_merged_dao_after_merger_combined_with_satstars_nmatch_cut_grade_a.fits')

  
    idx, d2d, d3d = nrca_catalog['skycoord'].match_to_catalog_sky(nrcb_catalog['skycoord'], nthneighbor=1)

    nrca_skycoord = nrca_catalog['skycoord']
    nrcb_skycoord = nrcb_catalog['skycoord']
    match_f140m = d2d < 0.05*u.arcsec

    new_nrcb = np.setdiff1d(np.arange(len(nrcb_skycoord)), idx[match_f140m])
    hdr = fits.getheader(image_filenames[filt], ext = ('SCI', 1))
    wcs = WCS(hdr)
    pixel_scale_deg2 = wcs.proj_plane_pixel_area()
    pixel_scale_arcsec = np.sqrt(pixel_scale_deg2) * 3600
    merged_catalog = vstack([nrca_catalog, nrcb_catalog[new_nrcb]])
    merged_catalog.meta['filter'] = filt
    merged_catalog.meta['pixelscale_deg2'] = pixel_scale_deg2
    merged_catalog.meta['pixelscale_arcsec'] = pixel_scale_arcsec

    merged_catalog.write(f'/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/{filt.lower()}_indivexp_merged_dao_after_merger_combined_with_satstars_nmatch_cut_grade_a.fits', overwrite=True)
    return merged_catalog

def merge_catalogs(tbls, tbls2=None, catalog_type='daophot',
                   ref_filter=None,
                   max_offset=0.10 * u.arcsec,
                   indivexp=False,
                   qfcut=None, fracfluxcut=None,
                   basepath='/orange/adamginsburg/w51/jwst/'):

    if tbls2 is None:
        tbls2 = []

    if len(tbls) == 0:
        raise ValueError("tbls must contain at least one table")

    all_tbls = list(tbls) + list(tbls2)

    if ref_filter is None:
        basetable_src = tbls[0]
        ref_filter = basetable_src.meta['filter']
    else:
        ref_filter_lower = ref_filter.lower()
        matches_ref = [tb for tb in tbls if tb.meta['filter'].lower() == ref_filter_lower]
        if len(matches_ref) == 0:
            raise ValueError("ref_filter must be present in tbls, not tbls2")
        basetable_src = matches_ref[0]
        ref_filter = basetable_src.meta['filter']

    print(f'ref_filter={ref_filter}', flush=True)
    print('tbls (group 1: allow new sources)', flush=True)
    for tb in tbls:
        tb.pprint(max_lines=10, max_width=-1)
        print(f'tb.meta={tb.meta}', flush=True)

    print('tbls2 (group 2: match only)', flush=True)
    for tb in tbls2:
        tb.pprint(max_lines=10, max_width=-1)
        print(f'tb.meta={tb.meta}', flush=True)

    basetable_src.meta['astrometric_reference_wavelength'] = ref_filter

    jfilts = SvoFps.get_filter_list('JWST')
    jfilts.add_index('filterID')

    reffiltercol = [ref_filter] * len(basetable_src)
    print(f"Started with {len(basetable_src)} in filter {ref_filter}", flush=True)

    # Build reference coordinates using only tbls (group 1)
    basecrds = basetable_src['skycoord']
    for tb in tqdm(tbls, desc='Group 1 Table Meta Loop'):
        if tb.meta['filter'].lower() == ref_filter.lower():
            continue

        crds = tb['skycoord']
        matches, sep, _ = crds.match_to_catalog_sky(basecrds, nthneighbor=1)
        reverse_matches, reverse_sep, _ = basecrds.match_to_catalog_sky(crds, nthneighbor=1)

        mutual_matches = (reverse_matches[matches] == np.arange(len(matches)))

        newcrds = crds[(sep > max_offset) | (~mutual_matches)]
        basecrds = SkyCoord([basecrds, newcrds])

        reffiltercol += [tb.meta['filter']] * len(newcrds)
        print(f"Added {len(newcrds)} new sources in filter {tb.meta['filter']}", flush=True)

    # tbls2 are intentionally NOT allowed to add new sources
    for tb in tqdm(tbls2, desc='Group 2 Table Meta Loop'):
        print(f"Matched-only filter {tb.meta['filter']}: no new sources will be added", flush=True)

    print(f"Base coordinate length = {len(basecrds)}", flush=True)

    basetable = Table()
    basetable['skycoord_ref'] = basecrds
    basetable['skycoord_ref_filtername'] = reffiltercol

    meta = {}

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        wls = []
        for tbl in tqdm(all_tbls, desc='Table Loop'):
            t0 = time.time()
            wl = tbl.meta['filter'].lower()
            wls.append(wl)

            crds = tbl['skycoord']
            matches, sep, _ = basecrds.match_to_catalog_sky(crds, nthneighbor=1)
            reverse_matches, reverse_sep, _ = crds.match_to_catalog_sky(basecrds, nthneighbor=1)

            mutual_matches = (reverse_matches[matches] == np.arange(len(matches)))

            print(f"filter {wl} has {len(tbl)} rows.  {mutual_matches.sum()} of {len(tbl)} are mutual.  Matching took {time.time()-t0:0.1f} seconds", flush=True)

            basetable.add_column(name=f"sep_{wl}", col=sep)
            basetable.add_column(name=f"id_{wl}", col=matches)

            matchtb = tbl[matches]
            badsep = sep > max_offset

            for cn in matchtb.colnames:
                if isinstance(matchtb[cn], SkyCoord):
                    matchtb.rename_column(cn, f"{cn}_{wl}")
                    matchtb[f'mask_{wl}'] = badsep | (~mutual_matches)
                else:
                    matchtb[f'{cn}_{wl}'] = MaskedColumn(data=matchtb[cn], name=f'{cn}_{wl}')
                    matchtb[f'{cn}_{wl}'].mask[badsep] = True
                    matchtb[f'{cn}_{wl}'].mask[~mutual_matches] = True
                    if hasattr(matchtb[cn], 'meta'):
                        matchtb[f'{cn}_{wl}'].meta = matchtb[cn].meta
                    matchtb.remove_column(cn)

            basetable = table.hstack([basetable, matchtb], join_type='exact')

            meta[f'{wl[1:-1]}pxdg'.upper()] = tbl.meta['pixelscale_deg2']
            meta[f'{wl[1:-1]}pxas'.upper()] = tbl.meta['pixelscale_arcsec']
            for key in tbl.meta:
                meta[f'{wl[1:-1]}{key[:4]}'.upper()] = tbl.meta[key]

        basetable.meta = meta

        nmatch_bands = np.zeros(len(basetable), dtype=int)
        for wl in wls:
            nmatch_bands += (basetable[f'flux_fit_{wl}'].mask == False).astype(int)

        basetable.add_column(name='nmatch_bands', col=nmatch_bands)

        tablename = f"{basepath}/catalogs/final_catalog_new"
        t0 = time.time()
        print(f"Writing table {tablename} with len={len(basetable)} and ncols={len(basetable.colnames)}", flush=True)
        basetable.meta['VERSION'] = datetime.datetime.now().isoformat()

        if os.path.exists(f"{tablename}.fits"):
            basetable.write(f"{tablename}.fits", overwrite=True)
        else:
            basetable.write(f"{tablename}.fits")

        print(f"Done writing table {tablename}.fits in {time.time()-t0:0.1f} seconds", flush=True)

    return basetable



 
       



In [2]:
f140m_catalog = merge_nrca_nrcb('f140m')
f162m_catalog = merge_nrca_nrcb('f162m')
f182m_catalog = merge_nrca_nrcb('f182m')
f187n_catalog = merge_nrca_nrcb('f187n')
f210m_catalog = merge_nrca_nrcb('f210m')
f335m_catalog = merge_nrca_nrcb('f335m')
f360m_catalog = merge_nrca_nrcb('f360m')
f405n_catalog = merge_nrca_nrcb('f405n')
f410m_catalog = merge_nrca_nrcb('f410m')
f480m_catalog = merge_nrca_nrcb('f480m')


f560w_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f560w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_nmatch_cut_grade_b_fixed.fits')
f770w_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f770w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_nmatch_cut_grade_b_fixed.fits')
f1000w_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f1000w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_nmatch_cut_grade_b_fixed.fits')
f1280w_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f1280w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_nmatch_cut_grade_b_fixed.fits')
f2100w_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f2100w_mirimage_indivexp_merged_dao_after_merger_combined_with_satstars_nmatch_cut_grade_b_fixed.fits')
f560w_catalog.meta['filter'] = 'f560w'
f560w_hdr = fits.getheader(image_filenames['f560w'], ext = ('SCI', 1))
f560w_wcs = WCS(f560w_hdr)
f560w_catalog.meta['pixelscale_deg2'] = f560w_wcs.proj_plane_pixel_area()
f560w_catalog.meta['pixelscale_arcsec'] = np.sqrt(f560w_catalog.meta['pixelscale_deg2']) * 3600
f770w_catalog.meta['filter'] = 'f770w'
f770w_hdr = fits.getheader(image_filenames['f770w'], ext = ('SCI', 1))
f770w_wcs = WCS(f770w_hdr)
f770w_catalog.meta['pixelscale_deg2'] = f770w_wcs.proj_plane_pixel_area()
f770w_catalog.meta['pixelscale_arcsec'] = np.sqrt(f770w_catalog.meta['pixelscale_deg2']) * 3600
f1000w_catalog.meta['filter'] = 'f1000w'
f1000w_hdr = fits.getheader(image_filenames['f1000w'], ext = ('SCI', 1))
f1000w_wcs = WCS(f1000w_hdr)
f1000w_catalog.meta['pixelscale_deg2'] = f1000w_wcs.proj_plane_pixel_area()
f1000w_catalog.meta['pixelscale_arcsec'] = np.sqrt(f1000w_catalog.meta['pixelscale_deg2']) * 3600
f1280w_catalog.meta['filter'] = 'f1280w'
f1280w_hdr = fits.getheader(image_filenames['f1280w'], ext = ('SCI', 1))
f1280w_wcs = WCS(f1280w_hdr)
f1280w_catalog.meta['pixelscale_deg2'] = f1280w_wcs.proj_plane_pixel_area()
f1280w_catalog.meta['pixelscale_arcsec'] = np.sqrt(f1280w_catalog.meta['pixelscale_deg2']) * 3600
f2100w_catalog.meta['filter'] = 'f2100w'
f2100w_hdr = fits.getheader(image_filenames['f2100w'], ext = ('SCI', 1))
f2100w_wcs = WCS(f2100w_hdr)
f2100w_catalog.meta['pixelscale_deg2'] = f2100w_wcs.proj_plane_pixel_area()
f2100w_catalog.meta['pixelscale_arcsec'] = np.sqrt(f2100w_catalog.meta['pixelscale_deg2']) * 3600

basetbl = [f140m_catalog, f162m_catalog, f182m_catalog, f187n_catalog, f210m_catalog, f335m_catalog, f360m_catalog, f405n_catalog, f410m_catalog, f480m_catalog, f560w_catalog, f770w_catalog, f1000w_catalog, f1280w_catalog, f2100w_catalog]
tabl2 = []

big_table = merge_catalogs(basetbl)


Set DATE-AVG to '2025-05-06T16:59:22.406' from MJD-AVG.
Set DATE-END to '2025-05-06T17:21:22.408' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.271881 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611441536.798 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2025-05-06T14:19:06.870' from MJD-AVG.
Set DATE-END to '2025-05-06T14:41:02.194' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.220336 from OBSGEO-[XYZ].
Set OBSGEO-H to 1610512569.543 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2025-05-06T15:11:59.611' from MJD-AVG.
Set DATE-END to '2025-05-06T15:35:36.878' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.237189 from OBSGEO-[XYZ].
Set OBSGEO-H to 1610816321.821 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2025-05-06T13:24:26.762' from MJD-AVG.
Set DATE-END to '2025-05-06T13:48:04.085' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.202551 from OBSGEO-[XYZ].
Set OBSGEO-H to 1610191901.140 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE

ref_filter=f140m
tbls (group 1: allow new sources)
     flux_fit           flux_err                     skycoord                       qfit                 cfit              local_bkg           roundness1          roundness2          sharpness      from_sat_catalog         std_ra                std_dec         nmatch nmatch_good   flux_err_prop   
                                                     deg,deg                                                                                                                                                                    deg                    deg                                                
------------------ ------------------ ------------------------------------- ------------------- ---------------------- ------------------ ------------------- -------------------- ------------------ ---------------- ---------------------- ---------------------- ------ ----------- ------------------
 36.22443969665813 0.6246585664655071 290.9307716733

Set DATE-AVG to '2024-09-08T11:47:44.336' from MJD-AVG.
Set DATE-END to '2024-09-08T13:00:37.070' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.791203 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291801554.144 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T11:55:48.768' from MJD-AVG.
Set DATE-END to '2024-09-08T13:08:42.701' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.785099 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291756623.231 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Started with 41122 in filter f140m


Group 1 Table Meta Loop:   0%|          | 0/15 [00:00<?, ?it/s]

Added 22339 new sources in filter f162m
Added 19841 new sources in filter f182m
Added 7243 new sources in filter f187n
Added 17364 new sources in filter f210m
Added 18626 new sources in filter f335m
Added 8733 new sources in filter f360m
Added 4582 new sources in filter f405n
Added 5222 new sources in filter f410m
Added 5022 new sources in filter f480m
Added 4699 new sources in filter f560w
Added 7465 new sources in filter f770w
Added 2230 new sources in filter f1000w
Added 1855 new sources in filter f1280w
Added 216 new sources in filter f2100w


Group 2 Table Meta Loop: 0it [00:00, ?it/s]

Base coordinate length = 166559


Table Loop:   0%|          | 0/15 [00:00<?, ?it/s]

filter f140m has 41122 rows.  41122 of 41122 are mutual.  Matching took 0.2 seconds
filter f162m has 52305 rows.  52164 of 52305 are mutual.  Matching took 0.2 seconds
filter f182m has 52576 rows.  52496 of 52576 are mutual.  Matching took 0.2 seconds
filter f187n has 29579 rows.  29522 of 29579 are mutual.  Matching took 0.2 seconds
filter f210m has 59882 rows.  59607 of 59882 are mutual.  Matching took 0.2 seconds
filter f335m has 39001 rows.  38993 of 39001 are mutual.  Matching took 0.2 seconds
filter f360m has 38179 rows.  38160 of 38179 are mutual.  Matching took 0.2 seconds
filter f405n has 19492 rows.  19482 of 19492 are mutual.  Matching took 0.1 seconds
filter f410m has 33705 rows.  33684 of 33705 are mutual.  Matching took 0.2 seconds
filter f480m has 26948 rows.  26933 of 26948 are mutual.  Matching took 0.2 seconds
filter f560w has 9001 rows.  8998 of 9001 are mutual.  Matching took 0.1 seconds
filter f770w has 11371 rows.  11366 of 11371 are mutual.  Matching took 0.1 sec